# Word embeddings from scratch (skip-gram + NS)

Interactive companion to `embeddings.py` / `run_smoke.py`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from embeddings import (
    TOY_CORPUS, SkipGramNS, build_vocab, corpus_token_ids,
    skipgram_pairs, unigram_noise_probs, nearest_neighbors, pca_2d,
)
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
word2id, id2word = build_vocab(TOY_CORPUS)
docs = corpus_token_ids(TOY_CORPUS, word2id)
pairs = skipgram_pairs(docs, window=2)
noise = unigram_noise_probs(docs, len(id2word))
print(f'vocab={len(id2word)} sentences={len(TOY_CORPUS)} pairs={len(pairs)}')

In [ ]:
model = SkipGramNS(vocab_size=len(id2word), dim=32, n_neg=5, lr=0.08, seed=42)
history = model.fit(pairs, noise, epochs=60)
emb = model.embeddings()
print(f'loss {history[0]:.4f} -> {history[-1]:.4f}')

plt.figure(figsize=(6,3))
plt.plot(history)
plt.xlabel('epoch'); plt.ylabel('loss'); plt.title('Skip-gram NS loss'); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
for q in ['cat', 'fly', 'king', 'python', 'sun', 'math']:
    if q in word2id:
        nn = nearest_neighbors(emb, word2id, id2word, q, k=5)
        print(q, '->', ', '.join(f'{w}({s:.3f})' for w, s in nn))

In [ ]:
xy = pca_2d(emb)
plt.figure(figsize=(8,6))
plt.scatter(xy[:,0], xy[:,1], c='lightgray', s=20)
for w in ['cat','dog','king','queen','python','java','sun','moon','fly','sky']:
    if w in word2id:
        i = word2id[w]
        plt.scatter(xy[i,0], xy[i,1], s=60)
        plt.annotate(w, (xy[i,0], xy[i,1]), fontsize=9)
plt.title('PCA of embeddings'); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.grid(True, alpha=0.3)
plt.show()